# Notebook: BCI_60_CarDet_Calc_Fec_Ini_Cd
*********************************************************************************


## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_60_CarDet_Calc_Fec_Ini_Cd.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011900796
* Autor: Gabriel Martínez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 23/05/2023
* Descripcion: Se obtiene la Fecha de Inicio de Cartera Deteriorada.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 08/07/2025 
* Descripción: Se modifica el proceso para incorporar una tabla de trabajo (work) que contiene el universo de clientes, incluyendo sus respectivas fechas de entrada a deterioro a nivel de cliente.
***************************************************************************

**************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 22/07/2025  
* Descripción: Se modificó completamente la lógica para calcular las operaciones y clientes que inician en deterioro, con el objetivo de mejorar la precisión del ingreso a cartera y ajustar el comportamiento histórico conforme a los criterios de negocio.     
***************************************************************************

**************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 13/08/2025  
* Descripción: Se modifico el campo rut del cliente que se esta seleccionando para la tabla temporal tmp_RES_tbl_dat_cli_ini_cd.
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_cartdet_stock
* {base_silver_x}.tbl_cd_cartdet_ope_ini_cd
* {base_silver_x}.tbl_cd_d00_segmentado
* {base_silver_x}.tbl_cd_cartdet_cli_ini_cd_pant
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_cartdet_ope_ini_cd
* {base_silver_x}.tbl_cd_cartdet_cli_ini_cd
***************************************************************************

## Carga Dependencias

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","03-Nombre BD Silver:")

fecha_x = dbutils.widgets.get("fecha_w")
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")


### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
#Parametria local notebook
p_cod_seg_ind='I'
p_cod_seg_gru='G'

print(f"p_cod_seg_ind: {p_cod_seg_ind}")
print(f"p_cod_seg_gru: {p_cod_seg_gru}")


In [0]:
resultado = obtener_parametros_tb_fecha(base_silver_x)
fecha_ant_y = resultado[1]  
print(f"[fecha_ant_y] {fecha_ant_y}")

### Extrae stock de operaciones deterioradas
--------------------------------------
- Extrae todos las operaciones deterioradas del proceso actual y sus motivos de deterioro


In [0]:
paso_query10 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cd_cartdet_stock AS
SELECT
	 periodo_cierre	
	,fecha_cierre	
	,tipo_proceso	
	,rut_cliente		
	,dv_rut_cliente	
	,operacion	
	,sistema		
	,segmento	
	,criterio_entrada
	,origen_deterioro
	,fecha_entrada	
FROM
	{base_silver_x}.tbl_cd_cartdet_stock
WHERE 
    fecha_cierre =   {fecha_x} 
"""

In [0]:
sql_safe(paso_query10)

### Obtenemos los datos de la tabla ope_ini_cd del periodo anterior.
--------------------------------------

In [0]:
paso_query20 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_dat_ope_ini_cd_PANT AS
SELECT  
  fecha_cierre,
  sistema,
  operacion,
  segmento,
  rut_cliente,
  fecha_entrada,
  cuotas_amort_por_vencer,
  saldo_capital_ifrs,
  origen_deterioro,
  fecha_informada
FROM
  {base_silver_x}.tbl_cd_cartdet_ope_ini_cd_pant
where
  fecha_informada = {fecha_ant_y}
"""

In [0]:
sql_safe(paso_query20)

### Extrae Operaciones D00 Periodo Actual
--------------------------------------
- Se extrae operaciones para periodo actual

In [0]:
paso_query25 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cd_d00_segmentado AS
SELECT
 A.periodo_cierre                                AS periodo_cierre,
 A.fecha_cierre                                  AS fecha_cierre,
 A.segmento                                      AS segmento,
 A.operacion                                     AS operacion,
 A.sistema                                       AS sistema,
 A.rut_cliente                                   AS rut_cliente,
 A.saldo_capital_ifrs                            AS saldo_capital_ifrs,
 A.cuotas_amort_por_vencer                       AS cuotas_amort_por_vencer
FROM
	{base_silver_x}.tbl_cd_d00_segmentado A 
WHERE 
    A.fecha_cierre = {fecha_x} 
"""

In [0]:
sql_safe(paso_query25)

mantener
### Obtenemos los datos de la tabla ope_ini_cd del periodo actual.
--------------------------------------

In [0]:
paso_query27 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_dat_ope_ini_cd_PACT AS
SELECT 
  a.periodo_cierre                          AS periodo_cierre,
  a.fecha_cierre                            AS fecha_cierre,
  a.operacion                               AS operacion ,
  a.sistema                                 AS sistema,
  a.segmento                                AS segmento,
  a.rut_cliente                             AS rut_cliente ,
  a.fecha_entrada                           AS fecha_entrada,
  a.criterio_entrada                        AS criterio_entrada,
  a.origen_deterioro                        AS origen_deterioro,
  COALESCE(c.saldo_capital_ifrs, 0)         AS saldo_capital_ifrs,
  COALESCE(c.cuotas_amort_por_vencer,0)     AS cuotas_amort_por_vencer,
  {fecha_x}                                 AS fecha_informada
FROM 
  tmp_EXT_tbl_cd_cartdet_stock a
LEFT JOIN 
  tmp_EXT_tbl_cd_d00_segmentado c
ON a.operacion = c.operacion   AND a.sistema = c.sistema
"""   

In [0]:
sql_safe(paso_query27)

### Generando los datos de la tabla ope_ini_cd.
--------------------------------------
* Si la operacion existe en el periodo anterior, se mantiene los datos del periodo anterior.
* Si la operacion NO existe en el periodo anterior 'es nueva', se ingresan los nuevos datos del periodo actual.

In [0]:
paso_query40 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_dat_ope_ini_cd AS
SELECT 
  a.periodo_cierre                                                   AS periodo_cierre,
  a.fecha_cierre                                                     AS fecha_cierre,
  a.operacion                                                        AS operacion ,
  a.sistema                                                          AS sistema,
  COALESCE(c.segmento,a.segmento)                                    AS segmento,
  COALESCE(c.rut_cliente,a.rut_cliente)                              AS rut_cliente ,
  COALESCE(c.fecha_entrada,a.fecha_entrada )                         AS fecha_entrada,
  COALESCE(c.origen_deterioro, a.origen_deterioro  )                 AS origen_deterioro,
  COALESCE(c.saldo_capital_ifrs, a.saldo_capital_ifrs)               AS saldo_capital_ifrs,
  COALESCE(c.cuotas_amort_por_vencer, a.cuotas_amort_por_vencer)     AS cuotas_amort_por_vencer,
  CASE WHEN c.operacion IS NULL THEN 1 ELSE 0 END                    AS ind_periodo, /* valor 1 significa que es nuevo */
  a.fecha_informada                                                  AS fecha_informada 
FROM 
  tmp_EXT_tbl_dat_ope_ini_cd_PACT a
LEFT JOIN 
  tmp_EXT_tbl_dat_ope_ini_cd_PANT c
ON  a.operacion = c.operacion AND a.sistema = c.sistema
"""   

In [0]:
sql_safe(paso_query40)

### Extrae Clientes deteriorados del periodo anterior 
--------------------------------------
- Extrae los clientes deteriorados del periodo anterior y la fecha de entrada a deterioro.


In [0]:
paso_query50 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_dat_cli_ini_cd_PANT AS
SELECT
   rut_cliente       AS rut_cliente
  ,MIN(fec_ini_cd)   AS fecha_entrada
  ,fecha_informada   AS fecha_informada
FROM
	{base_silver_x}.tbl_cd_cartdet_cli_ini_cd_pant
WHERE 
    fecha_informada =   {fecha_x}
GROUP BY 1,3    
"""

In [0]:
sql_safe(paso_query50)

### Calcula clientes y la fecha de Inicio de Deterioro.
--------------------------------------
- Se extraen los clientes y su minima fecha de inicio de cartera deteriorada desde la tabla dsr_gld_prodservicios_db.tbl_hcd_cartdet_cli_ini_cd

In [0]:
paso_query90 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_dat_cli_ini_cd_PACT AS
SELECT DISTINCT
    a.rut_cliente        AS rut_cliente,
    a.fecha_informada    AS fecha_informada,
    MIN(a.fecha_entrada) AS fecha_entrada  
FROM
    tmp_RES_tbl_dat_ope_ini_cd  a 
GROUP BY
    1,2
"""

In [0]:
sql_safe(paso_query90)

In [0]:
paso_query100 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_dat_cli_ini_cd AS
SELECT 
  a.rut_cliente                                  AS rut_cliente,
  COALESCE(c.fecha_entrada, a.fecha_entrada)     AS fecha_entrada,
  CASE WHEN c.rut_cliente IS NULL 
       THEN 1 
       ELSE 0 
  END AS ind_periodo, /* valor 1 significa que es nuevo */
  a.fecha_informada                              AS fecha_informada 
FROM 
  tmp_EXT_tbl_dat_cli_ini_cd_PACT a
LEFT JOIN 
  tmp_EXT_tbl_dat_cli_ini_cd_PANT c
ON 
  a.rut_cliente = c.rut_cliente
"""   

In [0]:
sql_safe(paso_query100)

## Carga Tablas de Salidas 
--------------------------------------
* carga resultados a tablas de salidas del notebook

#### Reproceso (Elimina registros en caso de reprocesos) 

##### TRUNCATE tbl_cd_cartdet_cli_ini_cd

In [0]:
paso_query200 = f"""TRUNCATE TABLE {base_silver_x}.tbl_cd_cartdet_cli_ini_cd """

In [0]:
sql_safe(paso_query200)

##### TRUNCATE tbl_cd_cartdet_ope_ini_cd

In [0]:

paso_query210 = f"""TRUNCATE TABLE {base_silver_x}.tbl_cd_cartdet_ope_ini_cd """

In [0]:
sql_safe(paso_query210)

#### Inserta Registros tabla salida

##### INSERT INTO tbl_cd_cartdet_cli_ini_cd

In [0]:
paso_query220 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_cli_ini_cd
SELECT 
    {fecha_x}          AS fecha_cierre,  
    A.rut_cliente        AS rut_cliente,
    A.fecha_entrada      AS fecha_entrada,  
    A.fecha_informada    AS fecha_informada
FROM
    tmp_RES_tbl_dat_cli_ini_cd A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.rut_cliente ORDER BY A.fecha_entrada DESC) =1    
"""


In [0]:
sql_safe(paso_query220)

##### INSERT INTO tbl_cd_cartdet_ope_ini_cd

In [0]:
paso_query230 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_ope_ini_cd
SELECT 
   A.fecha_cierre
  ,A.sistema
  ,A.operacion
  ,A.segmento
  ,A.rut_cliente
  ,A.fecha_entrada
  ,A.cuotas_amort_por_vencer
  ,A.saldo_capital_ifrs
  ,A.origen_deterioro
  ,A.fecha_informada
FROM 
  tmp_RES_tbl_dat_ope_ini_cd A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.operacion, A.sistema ORDER BY A.fecha_entrada DESC) =1     
"""


In [0]:
sql_safe(paso_query230)

## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")